In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 30 s


In [2]:
# STANDIN B2
# The three Part B models with the settings chosen on validation in B1 and B2
# (parts/10_baseline_models.ipynb). Dropped from the hand-in notebook, where the real B1 and B2 cells come first.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

tree = DecisionTreeClassifier(max_depth=12, random_state=SEED).fit(X_train, y_train)
logreg = Pipeline([("scale", StandardScaler()),
                   ("model", LogisticRegression(C=10.0, max_iter=2000, random_state=SEED))]).fit(X_train, y_train)
boost = make_boost().fit(X_train, y_train)
MODELS = {"tree": tree, "logreg": logreg, "boosting": boost}

<!-- STEP C1 -->
## Part C: learning with fewer labels

Labelling network flows is expensive: an analyst has to look at each one. Part C asks how good the
detector can be when only a small share of the training flows have a label, and whether the many
*unlabelled* flows can still help.

**Step C1: keep 10% of the labels.** We pretend that only 10% of the 183,063 training flows were
labelled by an analyst (18,306 flows, drawn at random but keeping every attack type's share). The other
90% (164,757 flows) keep their features but lose their labels. Their true labels are kept aside as
`y_hidden` **only to count, in C2, how many of the model's guesses were right**. They are never given to
a model to learn from.

**The lower line.** The black box (gradient boosting with the settings from B2) trained on the 18,306
labelled flows alone. Any method that also uses the unlabelled flows must beat this to count as help.
The **upper line** is the Part B model trained on all 183,063 labels. The validation and test sets are
the same as in Part B.

In [3]:
# STEP C1
# 10% of the TRAINING flows keep their label (stratified on the attack type, seed 42).
X_lab, X_unlab, y_lab, y_hidden = train_test_split(X_train, y_train, train_size=0.10,
                                                   stratify=type_train, random_state=SEED)
# y_hidden: the labels of the unlabelled 90%. Only used to count how many pseudo-labels are right
# (step C2). Never passed to .fit().

assert (len(X_lab), len(X_unlab)) == (18_306, 164_757), (len(X_lab), len(X_unlab))
assert X_lab.index.isin(X_train.index).all() and X_unlab.index.isin(X_train.index).all()
assert not X_lab.index.isin(X_val.index).any() and not X_lab.index.isin(X_test.index).any()
assert not X_lab.index.isin(X_unlab.index).any()

composition = pd.DataFrame({"labelled (10%)": type_train.loc[X_lab.index].value_counts(),
                            "unlabelled (90%)": type_train.loc[X_unlab.index].value_counts()})
composition.loc["Attack share"] = [y_lab.mean(), y_hidden.mean()]
print(f"Labelled: {len(X_lab):,} flows ({int(y_lab.sum()):,} attacks); "
      f"unlabelled: {len(X_unlab):,} flows")
composition.style.format(lambda v: f"{v:.2%}" if v < 1 else f"{v:,.0f}")

Labelled: 18,306 flows (4,815 attacks); unlabelled: 164,757 flows


,labelled (10%),unlabelled (90%)
attack_type,,
Benign,"13,491","121,418"
Exploits,"1,850","16,652"
Fuzzers,"1,561","14,045"
Reconnaissance,719,"6,473"
Generic,262,"2,361"
DoS,259,"2,330"
Shellcode,100,898
Backdoor,27,244
Analysis,23,208


In [4]:
# STEP C1
# The lower line: the same black box, trained on the 10% labelled flows only.
boost_few = make_boost().fit(X_lab, y_lab)

c1_val = pd.DataFrame({"lower line: boosting, 10% labels": score(boost_few, X_val, y_val),
                       "upper line: boosting, all labels": score(boost, X_val, y_val)}).T
c1_val = c1_val[["macro_f1", "recall", "pr_auc", "far", "accuracy"]]
c1_val.loc["difference (lower - upper)"] = c1_val.iloc[0] - c1_val.iloc[1]
c1_val.index.name = "model (validation set)"
c1_val.to_csv(TABLES / "C1_lower_line_val.csv", float_format="%.4f")
print(f"Lower line: {boost_few.n_iter_} trees")
c1_val.round(4)

Lower line: 82 trees


,macro_f1,recall,pr_auc,far,accuracy
model (validation set),,,,,
"lower line: boosting, 10% labels",0.9679,0.9862,0.9743,0.0296,0.9746
"upper line: boosting, all labels",0.9713,0.9924,0.9807,0.0282,0.9772
difference (lower - upper),-0.0034,-0.0062,-0.0064,0.0014,-0.0027


<!-- STEP C1 -->
### What the lower line shows

The labelled 10% (18,306 flows, 4,815 of them attacks) has the same mix of attack types as the full
training set. Trained on them alone, gradient boosting reaches a validation **macro-F1 of 0.9679,
recall 0.986, PR-AUC 0.974 and FAR 0.030**, against 0.9713 / 0.992 / 0.981 / 0.028 with all labels.

**Losing 90% of the labels costs very little**: 0.003 macro-F1 and 0.006 recall (on the 16,052
validation attacks, about 100 more missed). The lower line is still almost as good as the decision tree
trained on all labels (0.9685) and better than logistic regression on all labels (0.9608).

This sets the bar for step C2: the unlabelled 90% can win back **at most** the gap of 0.003 macro-F1
between the lower and the upper line. With 18,306 labelled flows the model already has plenty of examples
of every common attack type, so we expect pseudo-labelling to help little, if at all.